# Training a Multilayer Perceptron to Perform Image Classification for CUB_200_2011

This notebook attempts to train a Multilayer Perceptron to classify the 200 species of birds from their cropped images. It makes use of the preprocessing done by `notebooks/imagePrep.ipynb`, `scripts/prepare_images.py`, `notebooks/05_image_preparation.ipynb`, `scripts/augmentation.py`. As such, those files must be run in that order beforehand.

Table of Contents
1. Understanding our Data
2. Sourcing Image Data
3. Building the Model
4. Training
5. Evaluation and Results

> NOTE: Sections 1 and 2 have been copied over from "notebooks/efficientnetb0_fine_tuned.ipynb" due to shared image loading requirements.

In [1]:
# Setting constants to be used later
IMG_SIZE=224 # The width and height of the images
NUM_CLASSES=200 # The number of bird classes

## Section 1: Understanding our Data

Viewing the processed `cub_images_cropped.csv`.

In [2]:
import pandas as pd
img_data = pd.read_csv("../data/processed/cub_images_cropped.csv")

img_data.head()

,image_id,image_path,class_id,species,is_train,bbox_x,bbox_y,bbox_w,bbox_h,bbox_clipped,width,height,mode,cropped_path
0,1,001.Black_footed_Albatross/Black_Footed_Albatr...,1,Black footed Albatross,0,60.0,27.0,325.0,304.0,0,500,335,RGB,images_cropped/001.Black_footed_Albatross/Blac...
1,2,001.Black_footed_Albatross/Black_Footed_Albatr...,1,Black footed Albatross,1,139.0,30.0,153.0,264.0,0,500,336,RGB,images_cropped/001.Black_footed_Albatross/Blac...
2,3,001.Black_footed_Albatross/Black_Footed_Albatr...,1,Black footed Albatross,0,14.0,112.0,388.0,186.0,0,500,347,RGB,images_cropped/001.Black_footed_Albatross/Blac...
3,4,001.Black_footed_Albatross/Black_Footed_Albatr...,1,Black footed Albatross,1,112.0,90.0,255.0,242.0,0,415,500,RGB,images_cropped/001.Black_footed_Albatross/Blac...
4,5,001.Black_footed_Albatross/Black_Footed_Albatr...,1,Black footed Albatross,1,70.0,50.0,134.0,303.0,0,331,380,RGB,images_cropped/001.Black_footed_Albatross/Blac...


# Section 2: Sourcing Image Data

Instead of reworking the image loading logic that has already been coded in other files, this file will borrow their methods. Specifically, the following method adapts the image loading logic used in `notebooks/resnet50_altered_224px.ipynb` written by David Ling Jia Hao.

A separate function is made to convert from the PyTorch style loaders to more Tensorflow loaders. Tensorflow has been chosen for ease of use and familiarity as it was used during the lab sessions.

In [3]:
# The following code has been inspired by Claude
# It was given other files in this repo to figure out how to leverage them to load images
# Access that conversation here: https://claude.ai/share/fc7c1406-a1ea-49a5-9ec3-177e718d355e
# resnet50_altered_224px.ipynb was heavily referenced too.
# Despite the referencing, all code has been retyped by hand
# and edited in some places where required/desired by myself (Suhaib).

import sys
sys.path.insert(0, "../scripts")

import tensorflow as tf
from augmentation import make_loaders

train_dl, val_dl, test_dl = make_loaders("../data/processed", normalise=False)

# Getting a TF dataset (numpy) using David's loaders from PyTorch
def to_tf(loader):
    def gen():
        for x, y in loader:
            yield x.permute(0, 2, 3, 1).numpy() * 255.0, y.numpy()

    return tf.data.Dataset.from_generator(
        gen,
        output_signature=(tf.TensorSpec((None, IMG_SIZE, IMG_SIZE, 3), tf.float32),
                          tf.TensorSpec((None,), tf.int64))
    ).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = to_tf(train_dl), to_tf(val_dl), to_tf(test_dl)

print(train_ds)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


## Section 3: Building the Model

The following function has been inspired by [Fu (2026)](https://keras.io/examples/vision/image_classification_efficientnet_fine_tuning/) to build the fine-tuned model's architecture.

In [ ]:
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D 

def build_model():
    model = Sequential([
        Conv2D(64, kernel_size=(7, 7), activation="relu", input_shape=(224, 224, 3)),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25), # Regularisation
        Flatten(), # Flatten to be fed into a Dense layer
        Dense(512, activation="relu"),
        Dropout(0.5),
        Dense(384, activation="relu"),
        Dropout(0.2),
        Dense(200, activation="softmax")
    ])

    print(model.summary())
    return model


The following function has been inspired by [Fu (2026)](https://keras.io/examples/vision/image_classification_efficientnet_fine_tuning/) to plot historgrams.

In [24]:
# Function to visualise histogram after training
import matplotlib.pyplot as plt

def plot_hist(hist):
    plt.plot(hist.history["accuracy"], label="train accuracy")
    plt.plot(hist.history["val_accuracy"], label="val accuracy")
    plt.plot(hist.history["macro_f1"], label="train macro_f1")
    plt.plot(hist.history["val_macro_f1"], label="val macro_f1")
    plt.title("model accuracy")
    plt.ylabel("accuracy")
    plt.xlabel("epoch")
    plt.legend(loc="upper left")
    plt.show()